# BUBT-RGS Computer Vision Competition — RetinaMNIST

Final competition script from the provided source.

In [17]:
import os, sys, math, random, time, subprocess, csv, json
import numpy as np


def ensure(pkg):
    try:
        __import__(pkg.split("==")[0].replace("-", "_"))
        return
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


for pkg in ["medmnist>=3.0.1", "torchvision>=0.15.0", "scikit-learn>=1.2.0"]:
    ensure(pkg)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.metrics import (classification_report, confusion_matrix,
                              precision_recall_fscore_support, accuracy_score,
                              roc_auc_score, balanced_accuracy_score,
                              cohen_kappa_score)
import torchvision.transforms as T
from PIL import Image
import medmnist
from medmnist import INFO


# ---------------- Repro ---------------
def seed_all(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


seed_all(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# ---------------- Config ---------------
DATASET_FLAG = "retinamnist"
IMG_SIZE = 160
BATCH_SIZE = 128
NUM_WORKERS = 0
PIN_MEMORY = True
EPOCHS = 60
PATIENCE = 10
LR = 2e-4
WEIGHT_DECAY = 5e-4

USE_MIXED_PRECISION = (DEVICE == "cuda")
EMBED_DIM = 128
HEADS = 8
DIM_HEAD = 32
DEPTH = 2
PATCH = 2
BASE_CH = 32
DROPOUT = 0.1

IMBALANCE_BETA = 0.9999
FOCAL_GAMMA = 1.5
USE_WEIGHTED_SAMPLER = False
MIX_MODE = "both"
MIX_PROB = 0.3
MIX_ALPHA = 0.2
MINORITY_STRONG_P = 0.4

# ---- NEW: choose which loss to test (run BOTH across two experiments) ----
LOSS_TYPE = "ordinal_focal"  # "cb_focal" (organizers' original) or "ordinal_focal" (new)
ORDINAL_LAMBDA = 0.6     # weight of the ordinal-distance term vs class-balanced focal term

# ---- NEW: ensembling / TTA controls ----
TOP_K_CHECKPOINTS = 3
USE_TTA_AT_TEST = True

EXPERIMENT_NAME = f"{DATASET_FLAG}_{LOSS_TYPE}_seed42"
BEST_DIR = "checkpoints"
os.makedirs(BEST_DIR, exist_ok=True)

# ---------------- Data helpers ---------------
info = INFO[DATASET_FLAG]
Task = getattr(medmnist, info["python_class"])


def get_num_classes(info, Task):
    if "n_classes" in info:
        return int(info["n_classes"])
    if "label" in info and info["label"] is not None:
        try:
            return int(len(info["label"]))
        except Exception:
            pass
    ds = Task(split="train", download=True)
    labels = np.array(ds.labels).squeeze()
    return int(len(np.unique(labels)))


n_classes = get_num_classes(info, Task)
print("Classes:", n_classes)


def effective_num_class_weights(class_counts, beta=0.9999, eps=1e-12, normalize=True):
    counts = np.asarray(class_counts, dtype=np.float64)
    effective_num = 1.0 - np.power(beta, counts)
    weights = (1.0 - beta) / np.maximum(effective_num, eps)
    if normalize:
        weights = weights * (len(counts) / np.sum(weights))
    return torch.tensor(weights, dtype=torch.float32)


# ---------------- Augmentation: MixUp / CutMix ---------------
def mixup_data(x, y, alpha=0.3):
    if alpha <= 0:
        return x, y, None, 1.0
    lam = np.random.beta(alpha, alpha)
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam


def rand_bbox(W, H, lam):
    cut_rat = math.sqrt(1.0 - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)
    cx = np.random.randint(W)
    cy = np.random.randint(H)
    x1 = np.clip(cx - cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    y2 = np.clip(cy + cut_h // 2, 0, H)
    return x1, y1, x2, y2


def cutmix_data(x, y, alpha=0.3):
    if alpha <= 0:
        return x, y, None, 1.0
    lam = np.random.beta(alpha, alpha)
    B, C, H, W = x.size()
    index = torch.randperm(B, device=x.device)
    x1, y1, x2, y2 = rand_bbox(W, H, lam)
    x[:, :, y1:y2, x1:x2] = x[index, :, y1:y2, x1:x2]
    lam = 1.0 - ((x2 - x1) * (y2 - y1) / (W * H + 1e-8))
    y_a, y_b = y, y[index]
    return x, y_a, y_b, lam


def mix_batch(imgs, y, mode="both", p=0.7, alpha=0.3):
    if mode == "none" or np.random.rand() > p:
        return imgs, y, None, 1.0, "none"
    if mode == "mixup":
        mx = mixup_data(imgs, y, alpha)
        return (*mx, "mixup")
    if mode == "cutmix":
        cm = cutmix_data(imgs, y, alpha)
        return (*cm, "cutmix")
    if np.random.rand() < 0.5:
        mx = mixup_data(imgs, y, alpha)
        return (*mx, "mixup")
    else:
        cm = cutmix_data(imgs, y, alpha)
        return (*cm, "cutmix")


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


# ---------------- Minority-aware dataset wrapper ---------------
class ClassAwareTransformDataset(torch.utils.data.Dataset):
    def __init__(self, base_dataset, base_transform, strong_transform,
                 minority_classes, strong_p=0.8):
        self.base = base_dataset
        self.base_transform = base_transform
        self.strong_transform = strong_transform
        self.minority = set(int(c) for c in minority_classes)
        self.strong_p = float(strong_p)

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, target = self.base[idx]
        if not isinstance(img, Image.Image):
            img = Image.fromarray(img)
        y = int(np.array(target).squeeze())
        if (y in self.minority) and (np.random.rand() < self.strong_p):
            img = self.strong_transform(img)
        else:
            img = self.base_transform(img)
        return img, torch.tensor(y, dtype=torch.long)


train_tfms_base = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(0.5),
    T.RandomApply([T.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.95, 1.05))], p=0.7),
    T.ColorJitter(brightness=0.08, contrast=0.08, saturation=0.08, hue=0.01),
    T.ToTensor(),
    T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])
train_tfms_strong = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandAugment(num_ops=2, magnitude=7),
    T.RandomHorizontalFlip(0.5),
    T.RandomApply([T.RandomAffine(
        degrees=15,
        translate=(0.08, 0.08),
        scale=(0.92, 1.08)
    )], p=0.7),
    T.ToTensor(),
    T.Normalize([0.5,0.5,0.5], [0.5,0.5,0.5]),
])
val_tfms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])
# NEW: extra TTA views used only at final test time
tta_hflip = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)), T.RandomHorizontalFlip(1.0),
    T.ToTensor(), T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])
tta_rot = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)), T.RandomRotation((8, 8)),
    T.ToTensor(), T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

train_set_base = Task(split="train", transform=None, download=True, as_rgb=True)
val_set = Task(split="val", transform=val_tfms, download=True, as_rgb=True)
test_set = Task(split="test", transform=val_tfms, download=True, as_rgb=True)
test_set_raw = Task(split="test", transform=None, download=True, as_rgb=True)  # NEW: for TTA

labels_train = np.array(train_set_base.labels).squeeze().astype(int)
class_counts = np.bincount(labels_train, minlength=n_classes)
print("Class counts:", class_counts.tolist())

median_count = np.median(class_counts)
minority_classes = [i for i, c in enumerate(class_counts) if c < median_count]

train_set = ClassAwareTransformDataset(
    base_dataset=train_set_base,
    base_transform=train_tfms_base,
    strong_transform=train_tfms_strong,
    minority_classes=minority_classes,
    strong_p=MINORITY_STRONG_P
)

if USE_WEIGHTED_SAMPLER:
    cb_class_w = effective_num_class_weights(class_counts, beta=IMBALANCE_BETA)
    sample_weights = cb_class_w[torch.tensor(labels_train, dtype=torch.long)]
    train_sampler = torch.utils.data.WeightedRandomSampler(
        weights=sample_weights.double(), num_samples=len(sample_weights), replacement=True
    )
    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, sampler=train_sampler,
                               num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=False)
else:
    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=False)

val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=False)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=False)


# ---------------- Blocks (unchanged from organizers' starter) ---------------
class SEBlock(nn.Module):
    def __init__(self, ch, r=8):
        super().__init__()
        hidden = max(8, ch // r)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(ch, hidden, 1, bias=True), nn.SiLU(),
            nn.Conv2d(hidden, ch, 1, bias=True), nn.Sigmoid()
        )

    def forward(self, x):
        w = self.fc(self.pool(x))
        return x * w


class InvertedResidualSE(nn.Module):
    def __init__(self, in_ch, out_ch, expand_ratio=3, k=3, s=1, use_se=True):
        super().__init__()
        mid = int(in_ch * expand_ratio)
        self.use_res = (s == 1 and in_ch == out_ch)
        self.pw1 = nn.Sequential(nn.Conv2d(in_ch, mid, 1, bias=False), nn.BatchNorm2d(mid), nn.SiLU())
        self.dw = nn.Sequential(
            nn.Conv2d(mid, mid, k, stride=s, padding=k // 2, groups=mid, bias=False),
            nn.BatchNorm2d(mid), nn.SiLU()
        )
        self.se = SEBlock(mid) if use_se else nn.Identity()
        self.pw2 = nn.Sequential(nn.Conv2d(mid, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch))

    def forward(self, x):
        y = self.pw1(x); y = self.dw(y); y = self.se(y); y = self.pw2(y)
        return x + y if self.use_res else y


class FeatureExtractorDW(nn.Module):
    def __init__(self, in_ch=3, base=32, downsample=True):
        super().__init__()
        s2 = 2 if downsample else 1
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, base, kernel_size=3, stride=s2, padding=1, bias=False),
            nn.BatchNorm2d(base), nn.SiLU()
        )
        self.a1 = InvertedResidualSE(base, base, expand_ratio=3, k=5, s=1, use_se=True)
        self.a2 = InvertedResidualSE(base, base, expand_ratio=3, k=3, s=1, use_se=True)
        self.ds = InvertedResidualSE(base, base * 2, expand_ratio=3, k=5, s=2, use_se=True)
        self.b1 = InvertedResidualSE(base * 2, base * 2, expand_ratio=3, k=3, s=1, use_se=True)
        self.b2 = InvertedResidualSE(base * 2, base * 2, expand_ratio=3, k=5, s=1, use_se=True)
        self.refine = nn.Sequential(
            nn.Conv2d(base * 2, base * 2, kernel_size=3, padding=1, groups=base, bias=False),
            nn.BatchNorm2d(base * 2), nn.SiLU()
        )
        self.out_ch = base * 2

    def forward(self, x):
        x = self.stem(x)
        x = self.a1(x); x = self.a2(x)
        x = self.ds(x)
        x = self.b1(x); x = self.b2(x)
        x = self.refine(x)
        return x


class GhostModule(nn.Module):
    def __init__(self, in_ch, out_ch, ratio=2, kernel_size=1, dw_kernel=3, s=1, act=True):
        super().__init__()
        init_ch = math.ceil(out_ch / ratio)
        new_ch = init_ch * (ratio - 1)
        p = kernel_size // 2
        self.primary = nn.Sequential(
            nn.Conv2d(in_ch, init_ch, kernel_size, stride=s, padding=p, bias=False),
            nn.BatchNorm2d(init_ch), nn.SiLU() if act else nn.Identity()
        )
        self.cheap = nn.Sequential(
            nn.Conv2d(init_ch, new_ch, dw_kernel, stride=1, padding=dw_kernel // 2, groups=init_ch, bias=False),
            nn.BatchNorm2d(new_ch), nn.SiLU() if act else nn.Identity()
        )
        self.out_ch = out_ch

    def forward(self, x):
        y = self.primary(x); z = self.cheap(y)
        out = torch.cat([y, z], dim=1)
        return out[:, :self.out_ch, :, :]


class AdaptiveSelfAttention2d(nn.Module):
    def __init__(self, channels, max_heads=8, attn_dropout=0.1, proj_dropout=0.1, local_kernel=3, topk=3):
        super().__init__()
        self.channels = channels
        h = min(int(max_heads), channels)
        while h > 1 and (channels % h != 0):
            h -= 1
        self.heads = h
        self.dim_head = channels // h
        self.topk = int(topk)
        self.qkv = nn.Conv2d(channels, channels * 3, kernel_size=1, bias=True)
        self.q_norm = nn.LayerNorm(self.dim_head)
        self.k_norm = nn.LayerNorm(self.dim_head)
        self.talking_in = nn.Parameter(torch.eye(self.heads))
        self.talking_out = nn.Parameter(torch.eye(self.heads))
        self.tau = nn.Parameter(torch.ones(self.heads) * (self.dim_head ** -0.5))
        self.pos = nn.Conv2d(channels, channels, kernel_size=3, padding=1, groups=channels, bias=False)
        self.local = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=local_kernel, padding=local_kernel // 2,
                      groups=channels, bias=False),
            nn.Conv2d(channels, channels, kernel_size=1, bias=False), nn.SiLU()
        )
        self.alpha = nn.Parameter(torch.tensor(0.25))
        self.gamma = nn.Parameter(torch.tensor(0.5))
        self.attn_drop = nn.Dropout(attn_dropout)
        self.proj = nn.Conv2d(channels, channels, kernel_size=1, bias=False)
        self.proj_drop = nn.Dropout(proj_dropout)
        self.head_gate = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(channels, self.heads, kernel_size=1, bias=True))

    def forward(self, x):
        B, C, H, W = x.shape
        N = H * W
        x_pos = x + self.pos(x)
        qkv = self.qkv(x_pos)
        q, k, v = torch.chunk(qkv, 3, dim=1)

        def reshape(t):
            t = t.view(B, self.heads, self.dim_head, H, W)
            return t.flatten(-2).transpose(2, 3)

        q = reshape(q); k = reshape(k); v = reshape(v)
        q = self.q_norm(q.reshape(B * self.heads, N, self.dim_head)).reshape(B, self.heads, N, self.dim_head)
        k = self.k_norm(k.reshape(B * self.heads, N, self.dim_head)).reshape(B, self.heads, N, self.dim_head)
        attn = torch.einsum('b h n d, b h m d -> b h n m', q, k)
        attn = attn * self.tau.view(1, self.heads, 1, 1)
        attn = torch.einsum('hH, b h n m -> b H n m', self.talking_in, attn)
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        attn = torch.einsum('b h n m, hH -> b H n m', attn, self.talking_out)
        g_logits = self.head_gate(x).view(B, self.heads)
        g_probs = g_logits.softmax(dim=-1)
        if 1 <= self.topk < self.heads:
            topk_scores, topk_idx = torch.topk(g_probs, self.topk, dim=1)
            mask = torch.zeros(B, self.heads, device=x.device, dtype=attn.dtype)
            mask.scatter_(1, topk_idx, 1.0)
            head_weight = (mask * g_probs).view(B, self.heads, 1, 1)
        else:
            head_weight = g_probs.view(B, self.heads, 1, 1)
        attn = attn * head_weight
        out = torch.einsum('b h n m, b h m d -> b h n d', attn, v)
        out = out.transpose(2, 3).contiguous().view(B, C, H, W)
        out = self.proj_drop(self.proj(out))
        out = x + self.gamma * out + self.alpha * self.local(x)
        return out


class ABlock(nn.Module):
    def __init__(self, in_ch, base=32):
        super().__init__()
        self.rep_stage = FeatureExtractorDW(in_ch=in_ch, base=base, downsample=False)
        mid = self.rep_stage.out_ch
        self.adapt_attn = AdaptiveSelfAttention2d(mid, max_heads=8, attn_dropout=0.0, proj_dropout=0.0, topk=3)
        self.ghost = GhostModule(mid, mid, ratio=2, kernel_size=1, dw_kernel=3, s=1, act=True)
        self.proj = nn.Identity() if in_ch == mid else nn.Conv2d(in_ch, mid, 1, bias=False)
        self.bn = nn.BatchNorm2d(mid)
        self.act = nn.SiLU()
        self.out_ch = mid

    def forward(self, x):
        identity = self.proj(x)
        y = self.rep_stage(x)
        y = self.adapt_attn(y)
        y = self.ghost(y)
        if identity.shape[-2:] != y.shape[-2:]:
            identity = F.adaptive_avg_pool2d(identity, y.shape[-2:])
        return self.act(self.bn(y + identity))


class PatchEmbed(nn.Module):
    def __init__(self, in_ch, embed_dim, patch_size=2):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, embed_dim, kernel_size=patch_size, stride=patch_size, bias=False)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = self.proj(x)
        B, D, Hp, Wp = x.shape
        x = x.flatten(2).transpose(1, 2)
        x = self.norm(x)
        h = torch.linspace(0, 1, steps=Hp, device=x.device).repeat_interleave(Wp)
        w = torch.linspace(0, 1, steps=Wp, device=x.device).repeat(Hp)
        return x, h, w, Hp, Wp


def pos2d_sincos(Hp, Wp, dim, device):
    assert dim % 2 == 0
    dim_half = dim // 2

    def pe_1d(L, C):
        pos = torch.arange(L, device=device).unsqueeze(1)
        div = torch.exp(torch.arange(0, C, 2, device=device) * (-math.log(10000.0) / C))
        pe = torch.zeros(L, C, device=device)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        return pe

    py = pe_1d(Hp, dim_half)
    px = pe_1d(Wp, dim_half)
    pe = torch.zeros(Hp, Wp, dim, device=device)
    for i in range(Hp):
        for j in range(Wp):
            pe[i, j, :dim_half] = py[i]
            pe[i, j, dim_half:] = px[j]
    return pe.view(1, Hp * Wp, dim)


class LinearAttention(nn.Module):
    def __init__(self, dim, heads=4, dim_head=32, out_scale=0.5, eps=1e-6):
        super().__init__()
        self.heads = heads
        self.out_scale = out_scale
        self.eps = eps
        inner = heads * dim_head
        self.to_q = nn.Linear(dim, inner, bias=False)
        self.to_k = nn.Linear(dim, inner, bias=False)
        self.to_v = nn.Linear(dim, inner, bias=False)
        self.to_out = nn.Linear(inner, dim)
        self.scale = dim_head ** -0.5

    @staticmethod
    def phi(x):
        return F.elu(x) + 1.0

    def forward(self, x):
        B, N, D = x.shape
        H = self.heads

        def shape(proj):
            return proj.view(B, N, H, -1).transpose(1, 2)

        q = self.phi(shape(self.to_q(x))) * self.scale
        k = self.phi(shape(self.to_k(x)))
        v = shape(self.to_v(x))
        kv = k.transpose(-2, -1) @ v
        z = 1.0 / (q @ k.sum(dim=-2, keepdim=True).transpose(-2, -1) + self.eps)
        out = (q @ kv) * z
        out = out.transpose(1, 2).contiguous().view(B, N, -1)
        return self.to_out(out) * self.out_scale


class PlainFFN(nn.Module):
    def __init__(self, dim, hidden_mult=2, dropout=0.1):
        super().__init__()
        hidden = dim * hidden_mult
        self.net = nn.Sequential(
            nn.Linear(dim, hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(hidden, dim), nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class LiteBlock(nn.Module):
    def __init__(self, dim, heads=4, dim_head=32, hidden_mult=2, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = LinearAttention(dim, heads=heads, dim_head=dim_head, out_scale=0.5)
        self.norm2 = nn.LayerNorm(dim)
        self.ff = PlainFFN(dim, hidden_mult=hidden_mult, dropout=dropout)

    def forward(self, x, h_coord, w_coord):
        x = x + self.attn(self.norm1(x))
        x = x + self.ff(self.norm2(x))
        return x


class RetinaFixedNet(nn.Module):
    def __init__(self, in_ch=3, num_classes=5, base=32, embed_dim=128, patch=2,
                 heads=4, dim_head=32, depth=2, hidden_mult=2, dropout=0.1):
        super().__init__()
        self.stage1 = FeatureExtractorDW(in_ch=in_ch, base=base, downsample=True)
        self.ablock = ABlock(in_ch=self.stage1.out_ch, base=base)
        ch = self.ablock.out_ch
        self.patch = PatchEmbed(ch, embed_dim, patch_size=patch)
        self.blocks = nn.ModuleList([
            LiteBlock(embed_dim, heads=heads, dim_head=dim_head, hidden_mult=hidden_mult, dropout=dropout)
            for _ in range(depth)
        ])
        self.final_norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x = self.stage1(x)
        x = self.ablock(x)
        x, hcoord, wcoord, Hp, Wp = self.patch(x)
        x = x + pos2d_sincos(Hp, Wp, x.size(-1), x.device)
        for blk in self.blocks:
            x = blk(x, hcoord, wcoord)
        x = self.final_norm(x).mean(dim=1)
        return self.head(x)


# ---------------- Losses ---------------
class ClassBalancedFocalLoss(nn.Module):
    """Organizers' original loss — kept unchanged for the required A/B comparison."""

    def __init__(self, class_counts, beta=0.9999, gamma=2.0, smoothing=0.0, eps=1e-8):
        super().__init__()
        self.gamma = float(gamma)
        self.smoothing = float(smoothing)
        self.eps = eps
        w = effective_num_class_weights(class_counts, beta=beta, normalize=True)
        self.register_buffer("class_weights", w)

    def forward(self, logits, target):
        log_probs = F.log_softmax(logits, dim=1)
        probs = log_probs.exp().clamp_min(self.eps)
        n_classes_ = logits.size(1)
        cw = self.class_weights.to(logits.device)
        if self.smoothing > 0.0:
            with torch.no_grad():
                true_dist = torch.full_like(log_probs, self.smoothing / (n_classes_ - 1))
                true_dist.scatter_(1, target.view(-1, 1), 1.0 - self.smoothing)
            ce = -(true_dist * log_probs).sum(dim=1)
            pt = (true_dist * probs).sum(dim=1).clamp_min(self.eps)
            hard_w = cw[target]
            loss = hard_w * ((1.0 - pt) ** self.gamma) * ce
        else:
            ce = F.nll_loss(log_probs, target, reduction='none')
            pt = probs.gather(1, target.unsqueeze(1)).squeeze(1).clamp_min(self.eps)
            hard_w = cw[target]
            loss = hard_w * ((1.0 - pt) ** self.gamma) * ce
        return loss.mean()


class OrdinalSensitiveFocalLoss(nn.Module):
    """
    NEW (Stage E alternative). Same class-balanced focal base as the
    organizers' loss, PLUS an ordinal-distance penalty: predictions further
    (in grade) from the true label are penalized more, matching the
    clinical reality that DR severity is ordinal (grades progress 0->4).
    lambda_ord controls how much the ordinal term contributes.
    """

    def __init__(self, class_counts, num_classes, beta=0.9999, gamma=2.0,
                 lambda_ord=0.6, eps=1e-8):
        super().__init__()
        self.gamma = float(gamma)
        self.lambda_ord = float(lambda_ord)
        self.eps = eps
        self.num_classes = num_classes
        w = effective_num_class_weights(class_counts, beta=beta, normalize=True)
        self.register_buffer("class_weights", w)
        grades = torch.arange(num_classes, dtype=torch.float32)
        dist = (grades.view(-1, 1) - grades.view(1, -1)).abs()
        self.register_buffer("dist_matrix", dist)  # (C, C), dist_matrix[true, pred]

    def forward(self, logits, target):
        log_probs = F.log_softmax(logits, dim=1)
        probs = log_probs.exp().clamp_min(self.eps)
        cw = self.class_weights.to(logits.device)

        ce = F.nll_loss(log_probs, target, reduction='none')
        pt = probs.gather(1, target.unsqueeze(1)).squeeze(1).clamp_min(self.eps)
        hard_w = cw[target]
        focal_term = hard_w * ((1.0 - pt) ** self.gamma) * ce

        row_dist = self.dist_matrix[target]  # (B, C) distance of every class from the true grade
        expected_dist = (probs * row_dist).sum(dim=1)  # penalize probability mass on far-away grades
        max_dist = max(self.num_classes - 1, 1)
        ordinal_term = expected_dist / max_dist

        loss = focal_term + self.lambda_ord * ordinal_term
        return loss.mean()


# ---------------- Instantiate model ---------------
model = RetinaFixedNet(
    in_ch=3, num_classes=n_classes, base=BASE_CH,
    embed_dim=EMBED_DIM, patch=PATCH, heads=HEADS, dim_head=DIM_HEAD, depth=DEPTH,
    hidden_mult=2, dropout=DROPOUT
).to(DEVICE)

with torch.no_grad():
    if model.head.bias is not None:
        model.head.bias.zero_()

counts = class_counts.astype(np.float64)
priors = counts / np.maximum(counts.sum(), 1.0)

# ---------------- Optimizer & Scheduler ---------------
decay, no_decay = [], []
for n, p in model.named_parameters():
    if not p.requires_grad:
        continue
    nd = (n.endswith(".bias") or ("bn" in n.lower()) or ("norm" in n.lower()) or ("layernorm" in n.lower()))
    (no_decay if nd else decay).append(p)

optimizer = torch.optim.AdamW(
    [{"params": decay, "weight_decay": WEIGHT_DECAY}, {"params": no_decay, "weight_decay": 0.0}], lr=LR
)

from torch.optim.lr_scheduler import SequentialLR, LambdaLR, CosineAnnealingLR

warmup_epochs = 3
warmup = LambdaLR(optimizer, lr_lambda=lambda e: (e + 1) / warmup_epochs if e < warmup_epochs else 1.0)
cosine = CosineAnnealingLR(optimizer, T_max=EPOCHS - warmup_epochs, eta_min = 5e-6)
scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[warmup_epochs])

if LOSS_TYPE == "cb_focal":
    criterion = ClassBalancedFocalLoss(class_counts=class_counts, beta=IMBALANCE_BETA, gamma=FOCAL_GAMMA, smoothing=0.0)
elif LOSS_TYPE == "ordinal_focal":
    criterion = OrdinalSensitiveFocalLoss(class_counts=class_counts, num_classes=n_classes,
                                           beta=IMBALANCE_BETA, gamma=FOCAL_GAMMA, lambda_ord=ORDINAL_LAMBDA)
else:
    raise ValueError(f"Unknown LOSS_TYPE: {LOSS_TYPE}")
criterion = criterion.to(DEVICE)

scaler = torch.amp.GradScaler('cuda', enabled=USE_MIXED_PRECISION)
print(f"Params (M): {sum(p.numel() for p in model.parameters()) / 1e6:.2f}")

with torch.no_grad():
    model.eval()
    xdbg = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    f = model.stage1(xdbg); g = model.ablock(f)
    print("Feature map after A-block:", tuple(g.shape))
    z, h, w, Hp, Wp = model.patch(g)
    print("Tokens:", z.shape[1], "| Hp, Wp:", Hp, Wp)


# ---------------- Train / Evaluate ---------------
def evaluate(model, loader, criterion, logit_adjust=None):
    model.eval()
    total_loss = 0.0
    all_preds, all_targets, all_probs = [], [], []
    with torch.no_grad():
        for imgs, targets in loader:
            y = targets.squeeze().long().to(DEVICE)
            imgs = imgs.to(DEVICE, non_blocking=True)
            logits = model(imgs)
            loss = criterion(logits, y)
            total_loss += loss.item() * imgs.size(0)
            logits_adj = logits + (logit_adjust.view(1, -1) if logit_adjust is not None else 0.0)
            probs = logits_adj.softmax(dim=1).detach().cpu().numpy()
            preds = probs.argmax(axis=1).tolist()
            all_preds.extend(preds)
            all_targets.extend(y.detach().cpu().tolist())
            all_probs.append(probs)
    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_targets, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(all_targets, all_preds, average="macro", zero_division=0)
    balacc = balanced_accuracy_score(all_targets, all_preds)
    qwk = cohen_kappa_score(all_targets, all_preds, weights="quadratic")  # NEW: track ordinal agreement too
    all_probs = np.concatenate(all_probs, axis=0)
    try:
        auc = roc_auc_score(all_targets, all_probs, multi_class="ovr")
    except Exception:
        auc = float("nan")
    return avg_loss, acc, prec, rec, f1, balacc, auc, qwk, np.array(all_targets), np.array(all_preds)


def one_epoch(model, loader, criterion, optimizer=None,
              mix_mode=MIX_MODE, mix_prob=MIX_PROB, mix_alpha=MIX_ALPHA):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss = 0.0
    all_preds, all_targets = [], []
    for imgs, targets in loader:
        y = targets.squeeze().long().to(DEVICE)
        imgs = imgs.to(DEVICE, non_blocking=True)
        y_a = y_b = None
        lam = 1.0
        mix_tag = "none"
        if is_train and mix_mode != "none":
            imgs, y_a, y_b, lam, mix_tag = mix_batch(imgs, y, mode=mix_mode, p=mix_prob, alpha=mix_alpha)
        with torch.amp.autocast('cuda', enabled=USE_MIXED_PRECISION, dtype=torch.float16):
            logits = model(imgs)
            if is_train and mix_tag != "none":
                loss = mixup_criterion(criterion, logits, y_a, y_b, lam)
            else:
                loss = criterion(logits, y)
        if is_train:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=3.0)
            scaler.step(optimizer)
            scaler.update()
        total_loss += loss.item() * imgs.size(0)
        preds = logits.argmax(dim=1).detach().cpu().tolist()
        all_preds.extend(preds)
        all_targets.extend(y.detach().cpu().tolist())
    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_targets, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(all_targets, all_preds, average="macro", zero_division=0)
    return avg_loss, acc, prec, rec, f1


# ---------------- NEW: experiment logger (organizers require this record) ---------------
LOG_PATH = "experiment_log.csv"


def log_experiment(row: dict):
    file_exists = os.path.isfile(LOG_PATH)
    with open(LOG_PATH, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)


# ---------------- Train loop (BUG FIXED: real early stopping) ---------------
BEST_PATH = os.path.join(BEST_DIR, f"best_{EXPERIMENT_NAME}.pt")
best_score = -float("inf")
best_epoch = 0
patience = 0
# NEW: keep top-K checkpoints (by val score) for ensembling at the end
topk_records = []  # list of (score, path)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc, tr_p, tr_r, tr_f1 = one_epoch(
        model, train_loader, criterion, optimizer, mix_mode=MIX_MODE, mix_prob=MIX_PROB, mix_alpha=MIX_ALPHA
    )
    va_loss, va_acc, va_p, va_r, va_f1, va_balacc, va_auc, va_qwk, yv, yv_pred = evaluate(
        model, val_loader, criterion, logit_adjust=None
    )
    if scheduler is not None:
        scheduler.step()
        try:
            lr_now = scheduler.get_last_lr()[0]
        except Exception:
            lr_now = optimizer.param_groups[0]["lr"]
    else:
        lr_now = optimizer.param_groups[0]["lr"]

    score = 0.5 * va_f1 + 0.5 * va_balacc
    dt = time.time() - t0
    print(f"[Epoch {epoch:02d}/{EPOCHS}] "
          f"Train: loss={tr_loss:.4f} acc={tr_acc:.3f} F1={tr_f1:.3f} | "
          f"Val: loss={va_loss:.4f} acc={va_acc:.3f} F1={va_f1:.3f} "
          f"BalAcc={va_balacc:.3f} AUC={va_auc:.3f} QWK={va_qwk:.3f} | time={dt:.1f}s lr={lr_now:.2e}")

    if score > best_score + 1e-6:
        best_score = score
        best_epoch = epoch
        patience = 0
        ckpt_path = os.path.join(BEST_DIR, f"ckpt_ep{epoch:02d}_{EXPERIMENT_NAME}.pt")
        torch.save(model.state_dict(), ckpt_path)
        torch.save(model.state_dict(), BEST_PATH)
        topk_records.append((score, ckpt_path))
        topk_records = sorted(topk_records, key=lambda r: -r[0])[:TOP_K_CHECKPOINTS]
    else:
        patience += 1
        if patience >= PATIENCE:
            print(f"Early stopping triggered at epoch {epoch} (best epoch {best_epoch}, score={best_score:.3f}). "
                  f"Stopping now (bug fixed vs. organizers' original starter).")
            break  # <-- THE ACTUAL FIX: organizers' starter was missing this line

log_experiment({
    "experiment": EXPERIMENT_NAME,
    "loss_type": LOSS_TYPE,
    "img_size": IMG_SIZE, "batch_size": BATCH_SIZE, "epochs_run": epoch,
    "lr": LR, "weight_decay": WEIGHT_DECAY,
    "best_epoch": best_epoch, "best_val_score": round(best_score, 4),
    "params_M": round(sum(p.numel() for p in model.parameters()) / 1e6, 3),
    "seed": 42,
})
print(f"Logged this run to {LOG_PATH}")

# ---------------- Post-hoc calibration utilities ---------------
def collect_logits_targets(model, loader, device):
    model.eval()
    all_logits, all_targets = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.squeeze().long().to(device)
            logits = model(x)
            all_logits.append(logits.cpu())
            all_targets.append(y.cpu())
    return torch.cat(all_logits, 0), torch.cat(all_targets, 0)


def evaluate_with_bias_temp(logits, targets, bias_vec=None, T=1.0):
    z = logits.clone()
    if bias_vec is not None:
        z = z + bias_vec.view(1, -1)
    z = z / max(T, 1e-6)
    probs = z.softmax(dim=1).numpy()
    preds = probs.argmax(axis=1)
    acc = accuracy_score(targets, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(targets, preds, average="macro", zero_division=0)
    balacc = balanced_accuracy_score(targets, preds)
    qwk = cohen_kappa_score(targets, preds, weights="quadratic")
    try:
        auc = roc_auc_score(targets, probs, multi_class="ovr")
    except Exception:
        auc = float("nan")
    return acc, prec, rec, f1, balacc, auc, qwk, preds


def gridsearch_alpha_T(val_logits, val_targets, priors, alpha_grid=None, T_grid=None):
    if alpha_grid is None:
        alpha_grid = [0.0, 0.1, 0.2, 0.3, 0.4, 0.6, 0.8]
    if T_grid is None:
        T_grid = [0.7, 0.9, 1.0, 1.2, 1.5, 2.0]
    log_prior = torch.log(torch.clamp(torch.tensor(priors, dtype=torch.float32), 1e-8)).neg()
    best = {"score": -1.0, "alpha": 0.0, "T": 1.0, "bias": torch.zeros_like(log_prior)}
    for a in alpha_grid:
        bias = a * log_prior
        for T in T_grid:
            acc, pre, rec, f1, balacc, auc, qwk, _ = evaluate_with_bias_temp(val_logits, val_targets, bias, T)
            score = 0.5 * f1 + 0.5 * balacc
            if score > best["score"]:
                best = {"score": score, "alpha": a, "T": T, "bias": bias}
    return best


def load_weights_flex(model, path):
    ckpt = torch.load(path, map_location=DEVICE)
    try:
        model.load_state_dict(ckpt, strict=True)
    except Exception:
        def strip_module(k):
            return k.replace("module.", "", 1) if k.startswith("module.") else k
        model.load_state_dict({strip_module(k): v for k, v in ckpt.items()}, strict=False)


# ---------------- NEW: top-K checkpoint ensembling + TTA at test time ---------------
def ensemble_tta_predict(checkpoint_paths, raw_test_set, device):
    """Average softmax probabilities across checkpoints AND across TTA views."""
    tta_transforms = [val_tfms, tta_hflip, tta_rot]
    n = len(raw_test_set)
    probs_sum = torch.zeros(n, n_classes)
    targets = torch.zeros(n, dtype=torch.long)

    for ci, ckpt_path in enumerate(checkpoint_paths):
        load_weights_flex(model, ckpt_path)
        model.eval()
        for tf in tta_transforms:
            batch_imgs = []
            for i in range(n):
                img, label = raw_test_set[i]
                batch_imgs.append(tf(img))
                if ci == 0 and tf is tta_transforms[0]:
                    targets[i] = int(np.array(label).squeeze())
            batch = torch.stack(batch_imgs).to(device)
            with torch.no_grad():
                logits = model(batch)
                probs_sum += logits.softmax(dim=1).cpu()

    probs_avg = probs_sum / (len(checkpoint_paths) * len(tta_transforms))
    preds = probs_avg.argmax(dim=1).numpy()
    return targets.numpy(), preds, probs_avg.numpy()


# ---------------- Final Test Eval (baseline + calibrated + ensembled/TTA) ---------------
load_weights_flex(model, BEST_PATH)

te_loss, te_acc, te_p, te_r, te_f1, te_balacc, te_auc, te_qwk, yt, yp = evaluate(
    model, test_loader, criterion, logit_adjust=None
)
print("\n[FINAL TEST - baseline, single best checkpoint]")
print(f"loss={te_loss:.4f} | acc={te_acc:.3f} | F1={te_f1:.3f} | BalAcc={te_balacc:.3f} | "
      f"AUC(ovr)={te_auc:.3f} | QWK={te_qwk:.3f}")
print(classification_report(yt, yp, labels=list(range(n_classes)), zero_division=0, digits=3))
print(confusion_matrix(yt, yp, labels=list(range(n_classes))))

val_logits, val_targets = collect_logits_targets(model, val_loader, DEVICE)
best_cal = gridsearch_alpha_T(val_logits, val_targets, priors)
print(f"\n[CALIBRATION] best blended score={best_cal['score']:.3f} | alpha={best_cal['alpha']} | T={best_cal['T']}")

test_logits, test_targets = collect_logits_targets(model, test_loader, DEVICE)
te_acc_c, te_p_c, te_r_c, te_f1_c, te_balacc_c, te_auc_c, te_qwk_c, te_pred_c = evaluate_with_bias_temp(
    test_logits, test_targets, bias_vec=best_cal["bias"], T=best_cal["T"]
)
print("\n[FINAL TEST - calibrated, single best checkpoint]")
print(f"acc={te_acc_c:.3f} | F1={te_f1_c:.3f} | BalAcc={te_balacc_c:.3f} | AUC(ovr)={te_auc_c:.3f} | QWK={te_qwk_c:.3f}")

if USE_TTA_AT_TEST and len(topk_records) > 0:
    ckpt_paths = [p for _, p in topk_records]
    print(f"\nRunning ensemble+TTA over {len(ckpt_paths)} checkpoints: {ckpt_paths}")
    yt_e, yp_e, probs_e = ensemble_tta_predict(ckpt_paths, test_set_raw, DEVICE)
    acc_e = accuracy_score(yt_e, yp_e)
    prec_e, rec_e, f1_e, _ = precision_recall_fscore_support(yt_e, yp_e, average="macro", zero_division=0)
    balacc_e = balanced_accuracy_score(yt_e, yp_e)
    qwk_e = cohen_kappa_score(yt_e, yp_e, weights="quadratic")
    print("\n[FINAL TEST - top-K checkpoint ensemble + TTA]  <-- likely your best submission")
    print(f"acc={acc_e:.3f} | F1={f1_e:.3f} | BalAcc={balacc_e:.3f} | QWK={qwk_e:.3f}")
    print(classification_report(yt_e, yp_e, labels=list(range(n_classes)), zero_division=0, digits=3))
    print(confusion_matrix(yt_e, yp_e, labels=list(range(n_classes))))

    log_experiment({
        "experiment": EXPERIMENT_NAME + "_ensemble_tta",
        "loss_type": LOSS_TYPE, "img_size": IMG_SIZE, "batch_size": BATCH_SIZE,
        "epochs_run": epoch, "lr": LR, "weight_decay": WEIGHT_DECAY,
        "best_epoch": best_epoch, "best_val_score": round(best_score, 4),
        "params_M": round(sum(p.numel() for p in model.parameters()) / 1e6, 3),
        "seed": 42,
        "test_acc": round(acc_e, 4), "test_f1": round(f1_e, 4),
        "test_balacc": round(balacc_e, 4), "test_qwk": round(qwk_e, 4),
    })

print("\nDone. Remember: pick your final LOSS_TYPE using VALIDATION results, "
      "then report test numbers only once for your final chosen configuration.")

Device: cuda
Classes: 5
Class counts: [486, 128, 206, 194, 66]
Params (M): 0.70
Feature map after A-block: (1, 64, 20, 20)
Tokens: 100 | Hp, Wp: 10 10
[Epoch 01/60] Train: loss=1.0193 acc=0.142 F1=0.102 | Val: loss=1.0273 acc=0.075 F1=0.047 BalAcc=0.195 AUC=0.646 QWK=0.072 | time=127.4s lr=1.33e-04
[Epoch 02/60] Train: loss=0.9471 acc=0.392 F1=0.286 | Val: loss=1.0795 acc=0.050 F1=0.019 BalAcc=0.200 AUC=0.641 QWK=0.000 | time=9.8s lr=2.00e-04
[Epoch 03/60] Train: loss=0.8893 acc=0.467 F1=0.330 | Val: loss=1.0214 acc=0.167 F1=0.063 BalAcc=0.200 AUC=0.661 QWK=0.059 | time=10.2s lr=2.00e-04
[Epoch 04/60] Train: loss=0.8473 acc=0.496 F1=0.358 | Val: loss=0.9660 acc=0.233 F1=0.076 BalAcc=0.200 AUC=0.725 QWK=0.000 | time=10.1s lr=2.00e-04
[Epoch 05/60] Train: loss=0.8797 acc=0.390 F1=0.277 | Val: loss=0.8274 acc=0.433 F1=0.258 BalAcc=0.315 AUC=0.784 QWK=0.401 | time=9.5s lr=1.99e-04
[Epoch 06/60] Train: loss=0.8235 acc=0.470 F1=0.356 | Val: loss=0.7373 acc=0.517 F1=0.383 BalAcc=0.416 AUC=0.8